In [1]:
import pandas as pd
from unidecode import unidecode

customers = pd.read_csv("../data/raw/customers.csv", parse_dates = ['signup_date'], dtype = {'birth_year': 'Int64'})
products = pd.read_csv("../data/raw/products.csv", parse_dates = ['launch_date', 'discontinued_date'])
orders = pd.read_csv("../data/raw/orders.csv", parse_dates = ['order_ts', 'promised_date', 'delivered_date'], dtype = {'session_id' : 'Int64'})
orders.insert(4, "order_date_local", orders["order_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
order_items = pd.read_csv("../data/raw/order_items.csv", parse_dates = ['return_date'])
user_events = pd.read_csv("../data/raw/user_events.csv", parse_dates = ['event_ts'], dtype = {'customer_id': 'Int64', 'product_id': 'Int64'})
user_events.insert(6, "event_date_local", user_events["event_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
marketing_spend = pd.read_csv("../data/raw/marketing_spend.csv", parse_dates = ['spend_date'], dtype = {'platform_conversions' : 'Int64'})

tables = {"customers": customers,
          "products": products,
          "orders": orders,
          "order_items": order_items,
          "user_events": user_events,
          "marketing_spend": marketing_spend
         }

## Profiling function

In [2]:
def profile(df, name, key):
    print(f"===== {name} =====")
    print(f"\n***** Shape *****\n")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")
    print(f"***** Dtypes *****\n")
    print(df.dtypes)
    print(f"\n***** Unique Key *****\n")
    print(f"How many duplicates in {', '.join(key)}? {df.duplicated(subset=key).sum()}")
    print(f"How many missing values in {', '.join(key)}? {df[key].isna().any(axis=1).sum()}")
    print(f"How many fully duplicated rows? {df.duplicated().sum()}")
    print(f"\n***** NULL values in columns *****\n")
    display(pd.DataFrame({'Missing values': df.isna().sum(), '% Missing': (df.isna().mean() * 100).round(2)}))
    print(f"\n***** MIN, MAX, PERCENTILES *****\n")
    display(df.select_dtypes(include="number").describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)
    print(f"\n***** CATEGORICAL VARIABLES *****\n")
    
    text_cols = df.select_dtypes(exclude=["number", "datetime"]).columns
    for col in text_cols:
        n_unique = df[col].nunique()
        n_clean = df[col].str.strip().str.lower().nunique()
        print(f"\n----- {col}: {n_unique} unique -> {n_clean} after strip/lower -----\n")
        if n_unique <= 20:
            display(df[col].value_counts(dropna=False).to_frame("count"))
    print(f"\n***** DATE RANGE *****\n")
    display(df.select_dtypes(include="datetime").agg(["min", "max"]).T)

## CUSTOMERS

In [3]:
display(customers.head(10))

,customer_id,email_hash,signup_date,country,city,gender,birth_year,acquisition_source,marketing_opt_in
0,1,b57bf7094ed8,2023-12-28,LT,Test,NaN,<NA>,direct,0
1,2,8b7ec82f60c6,2024-01-01,LT,Šiauliai,Male,1999,organic_search,1
2,3,bbfae928f872,2024-01-02,LT,Klaipėda,Male,<NA>,direct,1
3,4,7548ae3fa95a,2024-01-02,LT,Kaunas,Female,1996,paid_social,1
4,5,528728837f93,2024-01-02,LT,Siauliai,Female,1991,paid_social,1
5,6,2ba1a69ceef4,2024-01-03,LT,Panevėžys,Female,1989,direct,0
6,7,7081b25e2e7e,2024-01-03,LT,Vilnius,Female,1979,email,0
7,8,d210317e87a9,2024-01-04,LT,Marijampolė,Male,1987,organic_search,1
8,9,c3c4cba41af4,2024-01-04,LT,Vilnius,Female,1993,direct,0
9,10,ffacef68e448,2024-01-05,EE,Narva,Male,1990,paid_search,1


In [4]:
profile(customers, "customers", ["customer_id"])

===== customers =====

***** Shape *****

Rows: 7483, Columns: 9

***** Dtypes *****

customer_id                    int64
email_hash                       str
signup_date           datetime64[us]
country                          str
city                             str
gender                           str
birth_year                     Int64
acquisition_source               str
marketing_opt_in               int64
dtype: object

***** Unique Key *****

How many duplicates in customer_id? 0
How many missing values in customer_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
customer_id,0,0.00
email_hash,0,0.00
signup_date,0,0.00
country,0,0.00
city,439,5.87
gender,483,6.45
birth_year,1009,13.48
acquisition_source,0,0.00
marketing_opt_in,0,0.00



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
customer_id,7483.0,3742.0,2160.300365,1.0,75.82,375.1,1871.5,3742.0,5612.5,7108.9,7408.18,7483.0
birth_year,6474.0,1984.959376,11.947787,1900.0,1955.73,1964.0,1977.0,1986.0,1994.0,2002.0,2006.0,2026.0
marketing_opt_in,7483.0,0.57664,0.494124,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0



***** CATEGORICAL VARIABLES *****


----- email_hash: 7427 unique -> 7427 after strip/lower -----


----- country: 3 unique -> 3 after strip/lower -----



...

,count
gender,
F,3107
M,2527
Female,685
Male,593
NaN,483
X,69
Other,19



----- acquisition_source: 7 unique -> 7 after strip/lower -----



,count
acquisition_source,
organic_search,2104
paid_social,1433
paid_search,1281
direct,1052
referral,646
affiliate,563
email,404



***** DATE RANGE *****



,min,max
signup_date,2023-12-28,2026-09-30


## My questions
1. Are all 'signup_date' values within the data period stated in the data dictionary?
2. Which values in 'gender' and 'city' mean the same thing but are written differently?
4. Can one person have more than one account (for example, the same email_hash)?

### Q1. Are all signup_date values within the data period stated in the data dictionary?

In [5]:
customers[customers["signup_date"] < "2024-01-01"]

,customer_id,email_hash,signup_date,country,city,gender,birth_year,acquisition_source,marketing_opt_in
0,1,b57bf7094ed8,2023-12-28,LT,Test,NaN,<NA>,direct,0


**Answer:** No. 1 customer (customer_id 1, the "Test" account) signed up on 2023-12-28, before the data period → DQ-01. All other signups are within the period.

### Q2. Which values in gender and city mean the same thing but are written differently?

In [6]:
customers_city = customers["city"].str.strip().str.lower().map(unidecode, na_action="ignore")
city_df = pd.DataFrame({"Raw": customers["city"], "Normalized": customers_city})

print(f"Unique cities: {customers['city'].nunique()} raw -> {customers_city.nunique()} normalized")

city_count = city_df.groupby("Normalized")["Raw"].nunique().sort_values(ascending=False)
print(city_count)

# Example: all spellings of Riga
city_df[city_df["Normalized"] == "riga"]["Raw"].unique()

Unique cities: 268 raw -> 130 normalized
Normalized
riga           8
marijampole    7
klaipeda       7
panevezys      7
siauliai       6
              ..
vinius         1
vlinius        1
vinlius        1
vlnius         1
vljandi        1
Name: Raw, Length: 130, dtype: int64


<StringArray>
['Riga', 'Rīga', 'Rīga ', 'Рига', ' Rīga', 'riga', 'rīga', 'RIGA']
Length: 8, dtype: str

**Answer:**
- **gender:** two encodings for the same values: F = Female, M = Male, X = Other (F/M/X: 5,703 rows, Female/Male/Other: 1,297 rows) → DQ-05.
- **city:** 268 raw values → 130 after strip, lower and removing accents. Types of variants:
  - case: RIGA / riga
  - extra spaces: "Rīga " / " Rīga"
  - missing local letters: Riga / Rīga, Siauliai / Šiauliai
  - Cyrillic: Рига
  - typos (not merged by normalisation): Vinius, Vlinius, Vlnius = Vilnius

  → DQ-03. A mapping to one correct name per city will be created in the SQL cleaning step (sql/01_clean.sql).

### Q3. Can one person have more than one account (for example, the same email_hash)?

In [7]:
duplicate_customers = customers[customers.duplicated(subset=["email_hash"], keep=False)].sort_values(by = ["email_hash", "customer_id"])
display(duplicate_customers.head(10))

print(f"\nRows of duplicates: {len(duplicate_customers)}")
print(f"\nHow many different 'email_hash': {duplicate_customers["email_hash"].nunique()}")

,customer_id,email_hash,signup_date,country,city,gender,birth_year,acquisition_source,marketing_opt_in
2553,2554,00b014275958,2025-02-21,LT,Panevėžys,F,1993,email,1
4279,4280,00b014275958,2025-10-02,LT,Panevezys,F,1993,email,1
1089,1090,08986822052b,2024-08-20,LT,NaN,Male,1975,referral,1
6110,6111,08986822052b,2026-04-21,LT,Klaipeda,M,1975,direct,1
649,650,0f8fb3021ac0,2024-06-06,LT,Kaunas,Female,1973,paid_search,1
2888,2889,0f8fb3021ac0,2025-04-07,LT,Kaunas,F,1973,paid_social,1
949,950,1e83c4eafbf5,2024-07-29,LT,kedainiai,Female,1974,paid_search,1
2854,2855,1e83c4eafbf5,2025-04-02,LT,Kėdainiai,F,1974,affiliate,1
4974,4975,1f09748adc87,2025-12-07,LT,Vilnius,NaN,1983,affiliate,1
6931,6932,1f09748adc87,2026-07-28,LT,Vilnius,F,<NA>,email,1



Rows of duplicates: 112

How many different 'email_hash': 56


**Answer:** Yes. 56 email_hash values are used by exactly 2 accounts each (112 rows). In the first 5 pairs, birth_year, country and city match, and the second account was created months later, so they look like the same person registering again → DQ-02.

## PRODUCTS

In [8]:
display(products.head(10))

,product_id,product_name,category,subcategory,brand,list_price,unit_cost,launch_date,discontinued_date
0,1,Lumea Reed Diffuser 100ml,Home & Kitchen,Decor,Lumea,19.99,84.00,2023-09-03,NaT
1,2,Strata Lightweight Running Jacket,Sports & Outdoors,Running,Strata,69.99,41.13,2023-09-03,NaT
2,3,Fjord Wear Men's Down Parka,Fashion,Outerwear,Fjord Wear,249.99,101.69,2023-09-04,NaT
3,4,Naturo Relaxing Massage Oil 200ml,Beauty & Care,Wellness,Naturo,14.99,8.56,2023-09-04,NaT
4,5,Pixora Indoor Security Camera 2K,Electronics,Smart Home,Pixora,49.99,35.03,2023-09-04,NaT
5,6,Silka Ionic Hair Dryer 2200W,Beauty & Care,Haircare,Silka,59.99,25.24,2023-09-04,NaT
6,7,Panela Stainless Steel Pot Set 5-Piece,Home & Kitchen,Cookware,Panela,139.99,72.36,2023-09-06,NaT
7,8,Whiskr Wet Cat Food 12x85g,Pets,Food,Whiskr,13.99,10.58,2023-09-06,NaT
8,9,Aktiva Football Size 5,Sports & Outdoors,Team Sports,Aktiva,24.99,15.40,2023-09-07,NaT
9,10,Dermia Men's Face Moisturizer 75ml,Beauty & Care,Skincare,Dermia,19.99,8.96,2023-09-07,NaT


In [9]:
profile(products, "products", ["product_id"])

===== products =====

***** Shape *****

Rows: 260, Columns: 9

***** Dtypes *****

product_id                    int64
product_name                    str
category                        str
subcategory                     str
brand                           str
list_price                  float64
unit_cost                   float64
launch_date          datetime64[us]
discontinued_date    datetime64[us]
dtype: object

***** Unique Key *****

How many duplicates in product_id? 0
How many missing values in product_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
product_id,0,0.00
product_name,0,0.00
category,0,0.00
subcategory,0,0.00
brand,0,0.00
list_price,0,0.00
unit_cost,0,0.00
launch_date,0,0.00
discontinued_date,232,89.23



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
product_id,260.0,130.500000,75.199734,1.00,3.5900,13.950,65.7500,130.50,195.2500,247.050,257.4100,260.00
list_price,260.0,66.555385,105.835678,6.99,8.9900,11.990,22.9900,34.99,69.9900,200.490,620.4900,949.99
unit_cost,260.0,43.057769,93.898865,2.56,3.9272,5.546,10.5375,19.09,37.7525,134.808,571.7689,852.20



***** CATEGORICAL VARIABLES *****


----- product_name: 260 unique -> 260 after strip/lower -----


----- category: 10 unique -> 9 after strip/lower -----



,count
category,
Fashion,50
Home & Kitchen,43
Sports & Outdoors,35
Beauty & Care,35
Electronics,29
Toys & Kids,25
Pets,20
Garden,20
Home and Kitchen,2



----- subcategory: 37 unique -> 37 after strip/lower -----


----- brand: 48 unique -> 48 after strip/lower -----


***** DATE RANGE *****



,min,max
launch_date,2023-09-03,2026-07-15
discontinued_date,2024-11-18,2026-07-10


## My questions
1. Are there products with 'unit_cost' equal or higher than 'list_price'? How many? Why?
2. Were any products sold before their 'launch_date' or after their 'discontinued_date'?
3. Are 'category', 'subcategory' and 'brand' values written consistently (no different spellings of the same value)?
4. Does every 'subcategory' belong to only one 'category'?

### Q1. Are there products with 'unit_cost' higher than 'list_price'? How many? Why?

In [10]:
# Which products have unit_cost equal to or higher than list_price?

bad_rows = products[products["unit_cost"] >= products["list_price"]]
display(bad_rows)

# Hypothesis: the decimal point in unit_cost is in the wrong place (cost is 10x too high).

# Check: compare the usual margin in the Decor subcategory with these two products' margins
# when unit_cost is divided by 10.

decor = products[products["subcategory"] == "Decor"]
decor["margin_pct"] = (decor["list_price"] - decor["unit_cost"]) / decor["list_price"] * 100
display(decor.sort_values("margin_pct"))
print(f"Subcategory's Decor median (excluding 2 bad rows) : {decor[decor["margin_pct"] >= 0]["margin_pct"].median().round(2)}")
bad_rows["new_unit_cost"] = bad_rows["unit_cost"] / 10
bad_rows["margin_pct"] = (bad_rows["list_price"] - bad_rows["new_unit_cost"]) / bad_rows["list_price"] * 100
display(bad_rows)



,product_id,product_name,category,subcategory,brand,list_price,unit_cost,launch_date,discontinued_date
0,1,Lumea Reed Diffuser 100ml,Home & Kitchen,Decor,Lumea,19.99,84.0,2023-09-03,NaT
241,242,Lumea Scented Candle Amber 200g,Home & Kitchen,Decor,Lumea,14.99,60.1,2026-04-09,NaT


,product_id,product_name,category,subcategory,brand,list_price,unit_cost,launch_date,discontinued_date,margin_pct
0,1,Lumea Reed Diffuser 100ml,Home & Kitchen,Decor,Lumea,19.99,84.00,2023-09-03,NaT,-320.210105
241,242,Lumea Scented Candle Amber 200g,Home & Kitchen,Decor,Lumea,14.99,60.10,2026-04-09,NaT,-300.933956
137,138,Verdane Artificial Olive Tree 120cm,Home & Kitchen,Decor,Verdane,64.99,31.57,2023-12-10,NaT,51.423296
65,66,Verdane Round Wall Mirror 60cm,Home & Kitchen,Decor,Verdane,69.99,33.83,2023-10-17,NaT,51.664524
25,26,Lumea LED Fairy Lights 10m,Home & Kitchen,Decor,Lumea,16.49,7.69,2023-09-20,NaT,53.365676
252,253,Verdane Ceramic Vase Large,Home & Kitchen,Decor,Verdane,34.99,15.41,2026-07-01,NaT,55.958845
110,111,Verdane Linen Cushion Cover 45x45,Home & Kitchen,Decor,Verdane,18.99,8.22,2023-11-26,NaT,56.714060
132,133,Verdane Wooden Photo Frame 20x30,Home and Kitchen,Decor,Verdane,14.99,6.08,2023-12-07,NaT,59.439626
190,191,Lumea Soy Wax Candle Set 3-Pack,Home & Kitchen,Decor,Lumea,27.99,10.62,2024-10-28,NaT,62.057878


Subcategory's Decor median (excluding 2 bad rows) : 55.96


,product_id,product_name,category,subcategory,brand,list_price,unit_cost,launch_date,discontinued_date,new_unit_cost,margin_pct
0,1,Lumea Reed Diffuser 100ml,Home & Kitchen,Decor,Lumea,19.99,84.0,2023-09-03,NaT,8.40,57.978989
241,242,Lumea Scented Candle Amber 200g,Home & Kitchen,Decor,Lumea,14.99,60.1,2026-04-09,NaT,6.01,59.906604


**Answer:** 2 products (product_id 1 and 242, both Lumea, Decor) have unit_cost about 4x higher than list_price. Other Decor products have a 51–62% margin (median 56%). If unit_cost is divided by 10, these two get 58% and 60%, which fits the normal range, so this is most likely a decimal error in unit_cost (10x too high) → DQ-07.

### Q2. Were any products sold before their 'launch_date' or after their 'discontinued_date'?

**Answer:** Not answered here. Sale dates are in other tables (order dates in orders, sold products in order_items), so this will be checked in notebooks/02_cross_table_checks.ipynb by joining the three tables and comparing each order date with the product's launch_date and discontinued_date.

### Q3. Are 'category', 'subcategory' and 'brand' values written consistently (no different spellings of the same value)?

**Answer:** No for category. Yes for subcategory and brand.
- category: "Electronics " has an extra space (1 product), and "Home and Kitchen" is used instead of "Home & Kitchen" (2 products) → DQ-08.
- subcategory and brand: no differences after strip/lower (37 and 48 values).

### Q4. Does every 'subcategory' belong to only one 'category'?

In [11]:
sub_cat_df = pd.DataFrame({"Subcategory" : products.groupby("subcategory")["category"].nunique(), "Categories" : products.groupby("subcategory")["category"].unique()})
display(sub_cat_df[sub_cat_df["Subcategory"] > 1])

,Subcategory,Categories
subcategory,,
Accessories,3,"[Fashion, Electronics, Electronics ]"
Decor,2,"[Home & Kitchen, Home and Kitchen]"
Textiles,2,"[Home & Kitchen, Home and Kitchen]"


**Answer:** Almost.
- Decor and Textiles show 2 categories only because of "Home and Kitchen" vs "Home & Kitchen" (DQ-08).
- Accessories is used in Fashion and Electronics. This is not an error, the products are different. Group by category + subcategory, not by subcategory alone.

## ORDERS

In [12]:
orders.head(10)

,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date
0,1,2,15,2024-01-01 14:18:20,2024-01-01,completed,NaN,bank_link,parcel_locker,2.99,27.98,2024-01-03,2024-01-03
1,2,1,32,2024-01-02 07:12:30,2024-01-02,cancelled,NaN,card,pickup,0.00,0.01,2024-01-02,NaT
2,3,1,34,2024-01-02 07:40:02,2024-01-02,cancelled,NaN,card,pickup,0.00,0.01,2024-01-02,NaT
3,4,3,39,2024-01-02 10:45:31,2024-01-02,completed,NaN,card,courier,0.00,144.98,2024-01-03,2024-01-03
4,5,4,45,2024-01-02 14:52:26,2024-01-02,completed,WELCOME10,card,parcel_locker,2.99,24.57,2024-01-04,2024-01-04
5,6,5,48,2024-01-02 17:06:21,2024-01-02,completed,WELCOME10,bank_link,parcel_locker,0.00,181.76,2024-01-04,2024-01-04
6,7,6,65,2024-01-03 00:48:14,2024-01-03,cancelled,WELCOME10,bank_link,courier,4.99,27.48,2024-01-04,NaT
7,8,1,81,2024-01-03 12:05:47,2024-01-03,completed,NaN,card,pickup,0.00,0.01,2024-01-04,2024-01-03
8,9,7,83,2024-01-03 12:12:26,2024-01-03,completed,NaN,card,courier,0.00,52.98,2024-01-05,2024-01-05
9,10,9,139,2024-01-04 15:35:51,2024-01-04,completed,NaN,card,parcel_locker,2.99,38.96,2024-01-06,2024-01-06


In [13]:
profile(orders, "orders", ["order_id"])

===== orders =====

***** Shape *****

Rows: 10269, Columns: 13

***** Dtypes *****

order_id                     int64
customer_id                  int64
session_id                   Int64
order_ts            datetime64[us]
order_date_local    datetime64[us]
status                         str
coupon_code                    str
payment_method                 str
delivery_method                str
shipping_fee               float64
order_total                float64
promised_date       datetime64[us]
delivered_date      datetime64[us]
dtype: object

***** Unique Key *****

How many duplicates in order_id? 10
How many missing values in order_id? 0
How many fully duplicated rows? 10

***** NULL values in columns *****



,Missing values,% Missing
order_id,0,0.00
customer_id,0,0.00
session_id,584,5.69
order_ts,0,0.00
order_date_local,0,0.00
status,0,0.00
coupon_code,8411,81.91
payment_method,0,0.00
delivery_method,0,0.00
shipping_fee,0,0.00



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
order_id,10269.0,5129.646996,2960.241604,1.0,103.68,514.4,2568.0,5125.0,7692.0,9745.6,10156.32,10259.0
customer_id,10269.0,3333.881196,2100.220309,1.0,61.0,308.4,1486.0,3161.0,5071.0,6894.6,7346.32,7482.0
session_id,9685.0,105026.223851,60290.575041,15.0,2141.88,10397.8,52473.0,107133.0,156546.0,199116.4,207423.64,209303.0
shipping_fee,10269.0,2.018672,2.284789,0.0,0.0,0.0,0.0,0.0,3.49,5.49,7.49,7.49
order_total,10269.0,72.027268,101.937539,0.01,11.98,15.18,27.98,46.98,78.27,193.282,612.4864,2018.97



***** CATEGORICAL VARIABLES *****


----- status: 4 unique -> 4 after strip/lower -----



...

,count
payment_method,
bank_link,4618
card,4530
paypal,581
pay_later,430
cash_on_delivery,110



----- delivery_method: 3 unique -> 3 after strip/lower -----



,count
delivery_method,
parcel_locker,6310
courier,3195
pickup,764



***** DATE RANGE *****



,min,max
order_ts,2024-01-01 14:18:20,2026-09-30 19:03:40
order_date_local,2024-01-01 00:00:00,2026-09-30 00:00:00
promised_date,2024-01-02 00:00:00,2026-10-03 00:00:00
delivered_date,2024-01-03 00:00:00,2026-09-30 00:00:00


## My questions
1. Are all 'order_ts', 'promised_date', 'delivered_date' values within the data period stated in the data dictionary? ('order_ts' is UTC time)
2. Are there any cancelled orders that have a 'delivered_date'?
3. Are there any orders whose local order date is later than 'promised_date' or 'delivered_date'?
4. Are there any orders that don't look like real customer orders ('order_total' close to 0)? Who placed them?

### Q1. Are all 'order_ts', 'promised_date', 'delivered_date' values within the data period stated in the data dictionary? ('order_ts' is UTC time)

**Answer:** Yes (from profile()):

| Column | Min | Max |
|---|---|---|
| order_date_local | 2024-01-01 | 2026-09-30 |
| promised_date | 2024-01-02 | 2026-10-03 |
| delivered_date | 2024-01-03 | 2026-09-30 |

promised_date goes past 2026-09-30, but this is normal: late September orders were promised for early October.

### Q2. Are there any cancelled orders that have a 'delivered_date'?

In [14]:
print(f"Orders which were cancelled but have 'delivered_date': {len((orders[(orders["status"] == "cancelled") & (orders["delivered_date"].notnull())]))}")

Orders which were cancelled but have 'delivered_date': 0


**Answer:** No. 0 cancelled orders have a delivered_date. All 304 empty delivered_date values belong to cancelled (272), shipped (21) or processing (11) orders, which is expected.

### Q3. Are there any orders whose 'order_date_local' is later than 'promised_date' or 'delivered_date'?

In [15]:
bad_date_orders = orders[(orders["order_date_local"] > orders["promised_date"]) | (orders["order_date_local"] > orders["delivered_date"])]
print(f"Bad date orders: {len(bad_date_orders)}")
display(bad_date_orders)

Bad date orders: 15


,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date
518,519,529,10528,2024-05-15 21:30:25,2024-05-16,completed,WELCOME10,bank_link,parcel_locker,0.00,58.48,2024-05-17,2024-05-15
3680,3681,1098,74877,2025-04-29 12:01:00,2025-04-29,completed,NaN,card,pickup,0.00,12.99,2025-04-30,2025-04-28
4187,4188,3426,85366,2025-06-13 10:11:30,2025-06-13,completed,WELCOME10,card,pickup,0.00,11.69,2025-06-13,2025-06-11
5131,5132,4086,107360,2025-09-11 03:08:15,2025-09-11,completed,NaN,bank_link,parcel_locker,3.49,27.47,2025-09-12,2025-09-09
5708,5709,3221,118950,2025-10-28 06:13:20,2025-10-28,completed,NaN,bank_link,pickup,0.00,29.99,2025-10-28,2025-10-27
5719,5720,3124,119245,2025-10-29 06:49:17,2025-10-29,completed,NaN,card,parcel_locker,4.49,16.48,2025-10-31,2025-10-27
7010,7011,5373,143042,2026-01-20 04:24:30,2026-01-20,completed,NaN,card,parcel_locker,3.49,15.48,2026-01-21,2026-01-19
7328,7329,5381,149310,2026-02-17 14:18:31,2026-02-17,completed,NaN,card,parcel_locker,3.49,26.48,2026-02-19,2026-02-16
7576,7577,3111,154203,2026-03-10 14:29:05,2026-03-10,completed,NaN,paypal,parcel_locker,0.00,61.97,2026-03-13,2026-03-08
7739,7740,3375,157592,2026-03-23 18:59:05,2026-03-23,completed,NaN,bank_link,parcel_locker,0.00,196.95,2026-03-25,2026-03-21


**Answer:** Yes. 15 orders have delivered_date 1–2 days before order_date_local, which is impossible. promised_date is fine in all of them → DQ-18.

### Q4. Are there any orders that don't look like real customer orders ('order_total' close to 0)? Who placed them?

In [16]:
fake_customer_orders = orders.copy()
fake_customer_orders["net_total"] = orders["order_total"] - orders["shipping_fee"]

display(fake_customer_orders.nsmallest(15, "net_total"))

,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date,net_total
1239,1240,1175,26175,2024-09-03 07:42:11,2024-09-03,completed,NaN,bank_link,courier,4.99,4.99,2024-09-04,2024-09-04,0.00
1,2,1,32,2024-01-02 07:12:30,2024-01-02,cancelled,NaN,card,pickup,0.00,0.01,2024-01-02,NaT,0.01
2,3,1,34,2024-01-02 07:40:02,2024-01-02,cancelled,NaN,card,pickup,0.00,0.01,2024-01-02,NaT,0.01
7,8,1,81,2024-01-03 12:05:47,2024-01-03,completed,NaN,card,pickup,0.00,0.01,2024-01-04,2024-01-03,0.01
2570,2571,2131,52576,2025-01-09 14:12:24,2025-01-09,completed,WELCOME10,bank_link,parcel_locker,2.99,8.83,2025-01-11,2025-01-11,5.84
4898,4899,3935,101866,2025-08-19 05:42:10,2025-08-19,completed,WELCOME10,paypal,parcel_locker,3.49,9.33,2025-08-20,2025-08-20,5.84
57,58,61,1265,2024-01-28 19:52:33,2024-01-28,completed,WELCOME10,card,parcel_locker,2.99,9.28,2024-01-30,2024-01-30,6.29
9911,9912,7268,202171,2026-09-05 08:23:45,2026-09-05,completed,WELCOME10,card,courier,7.49,13.78,2026-09-09,2026-09-09,6.29
9551,9552,7045,195009,2026-08-10 13:06:25,2026-08-10,completed,WELCOME10,card,pickup,0.00,6.29,2026-08-11,2026-08-11,6.29
1968,1969,1118,40735,2024-11-19 12:59:56,2024-11-19,completed,NaN,card,parcel_locker,2.99,9.48,2024-11-21,2024-11-21,6.49


**Answer:** Yes, 4 orders. Item value (order_total - shipping_fee) jumps from 0.01 straight to 5.84, so only these 4 look unreal:
- orders 2, 3, 8 (0.01 each) were placed by customer_id 1, the "Test" account (DQ-01) → test orders, DQ-10.
- order 1240 (items 0.00, only shipping paid) was placed by customer_id 1175 → DQ-13.

## ORDER_ITEMS

In [17]:
order_items.head(10)

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,is_returned,return_reason,return_date
0,1,1,31,1,24.99,0,0,NaN,NaT
1,2,2,111,1,0.01,0,0,NaN,NaT
2,3,3,136,1,0.01,0,0,NaN,NaT
3,4,4,7,1,119.99,0,0,NaN,NaT
4,5,4,85,1,24.99,0,0,NaN,NaT
5,6,5,78,1,11.99,10,0,NaN,NaT
6,7,5,132,1,11.99,10,0,NaN,NaT
7,8,6,7,1,119.99,10,0,NaN,NaT
8,9,6,41,1,19.99,10,1,changed_mind,2024-01-06
9,10,6,109,1,44.99,10,0,NaN,NaT


In [18]:
profile(order_items, "order_items", ["order_item_id"])

===== order_items =====

***** Shape *****

Rows: 17784, Columns: 9

***** Dtypes *****

order_item_id             int64
order_id                  int64
product_id                int64
quantity                  int64
unit_price              float64
discount_pct              int64
is_returned               int64
return_reason               str
return_date      datetime64[us]
dtype: object

***** Unique Key *****

How many duplicates in order_item_id? 0
How many missing values in order_item_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
order_item_id,0,0.00
order_id,0,0.00
product_id,0,0.00
quantity,0,0.00
unit_price,0,0.00
discount_pct,0,0.00
is_returned,0,0.00
return_reason,16312,91.72
return_date,16218,91.19



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
order_item_id,17784.0,8892.500000,5133.942929,1.0,178.83,890.15,4446.75,8892.50,13338.25,16894.85,17606.17,17784.00
order_id,17784.0,5108.625337,2948.382302,1.0,99.83,522.15,2571.75,5110.00,7649.25,9740.00,10146.17,10259.00
product_id,17784.0,110.092780,62.689397,1.0,4.00,14.00,58.00,110.00,153.00,224.00,244.17,260.00
quantity,17784.0,1.086651,0.387445,1.0,1.00,1.00,1.00,1.00,1.00,2.00,3.00,15.00
unit_price,17784.0,39.933268,68.447364,0.0,8.49,10.49,14.99,27.99,39.99,99.99,259.99,1099.99
discount_pct,17784.0,4.712382,9.022094,0.0,0.00,0.00,0.00,0.00,10.00,25.00,40.00,50.00
is_returned,17784.0,0.088057,0.283385,0.0,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00



***** CATEGORICAL VARIABLES *****


----- return_reason: 7 unique -> 7 after strip/lower -----



,count
return_reason,
NaN,16312
changed_mind,416
not_as_described,284
defective,249
size_too_small,198
size_too_large,172
arrived_late,84
wrong_item,69



***** DATE RANGE *****



,min,max
return_date,2024-01-06,2026-09-30


## My questions
1. Can the same 'product_id' appear more than once within one 'order_id'?
2. Are 'quantity', 'unit_price' and 'discount_pct' within realistic ranges?
3. When 'is_returned' = 1, are 'return_reason' and 'return_date' always filled in? When 'is_returned' = 0, are they always empty?

### Q1. Can the same 'product_id' appear more than once within one 'order_id'?

In [19]:
repeating_products = order_items[order_items.duplicated(subset = ["product_id", "order_id"], keep = False)].sort_values(by = ["product_id", "order_id"])
display(repeating_products)


,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,is_returned,return_reason,return_date


**Answer:** No. Each product appears only once per order (0 duplicates of order_id + product_id). This combination can be used as a logical key for order lines.

### Q2. Are 'quantity', 'unit_price' and 'discount_pct' within realistic ranges?

In [20]:
high_quantity = order_items[order_items["quantity"] > 3]
display(high_quantity)

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,is_returned,return_reason,return_date
2147,2148,1240,223,12,0.00,0,0,NaN,NaT
2148,2149,1240,259,10,0.00,0,0,NaN,NaT
5593,5594,3220,33,15,34.99,0,0,NaN,NaT
9632,9633,5516,95,10,29.99,0,0,NaN,NaT
9701,9702,5558,111,10,18.99,0,0,NaN,NaT
9961,9962,5708,26,10,14.99,0,0,NaN,NaT
14671,14672,8406,31,12,29.99,0,0,NaN,NaT
14672,14673,8406,21,12,24.99,0,0,NaN,NaT


**Answer:** Mostly yes.
- discount_pct: 0–50%, realistic for sales.
- unit_price: realistic, except 2 lines at 0.00 in order 1240 → DQ-13.
- quantity: 99% of lines have 3 or fewer; 8 lines have 10–15 units. Some look like business orders, others could be typing errors → DQ-12, will be checked against order_total in 02 notebook.

### Q3. When 'is_returned' = 1, are 'return_reason' and 'return_date' always filled in? When 'is_returned' = 0, are they always empty?

In [21]:
check_1 = order_items[(order_items["is_returned"] == 1) & (order_items["return_reason"].isnull())]
check_2 = order_items[(order_items["is_returned"] == 1) & (order_items["return_date"].isnull())]
check_3 = order_items[(order_items["is_returned"] == 0) & (order_items["return_reason"].notnull())]
check_4 = order_items[(order_items["is_returned"] == 0) & (order_items["return_date"].notnull())]
                                                        
print(f"""
Returned, no reason: {len(check_1)}
Returned, no date:   {len(check_2)}
Not returned, return reason: {len(check_3)}
Not returned, return date:   {len(check_4)}""")


Returned, no reason: 94
Returned, no date:   0
Not returned, return reason: 0
Not returned, return date:   0


**Answer:**
- is_returned = 1: return_date is always filled in, but 94 lines have no return_reason → DQ-11.
- is_returned = 0: return_reason and return_date are always empty.

## Check: which lines have the lowest unit_price? (min was 0.00 in profile)

In [22]:
order_items.nsmallest(6, "unit_price")

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,is_returned,return_reason,return_date
2147,2148,1240,223,12,0.00,0,0,NaN,NaT
2148,2149,1240,259,10,0.00,0,0,NaN,NaT
1,2,2,111,1,0.01,0,0,NaN,NaT
2,3,3,136,1,0.01,0,0,NaN,NaT
12,13,8,121,1,0.01,0,0,NaN,NaT
360,361,208,117,1,6.49,10,0,NaN,NaT


## USER_EVENTS

In [23]:
user_events.head(10)

,event_id,session_id,visitor_id,customer_id,event_type,event_ts,event_date_local,device,traffic_source,product_id
0,1,1,7940527645,<NA>,view_item,2024-01-01 05:15:03,2024-01-01,mobile,organic_search,109
1,2,1,7940527645,<NA>,view_item,2024-01-01 05:18:46,2024-01-01,mobile,organic_search,111
2,3,1,7940527645,<NA>,view_item,2024-01-01 05:18:52,2024-01-01,mobile,organic_search,41
3,4,1,7940527645,<NA>,view_item,2024-01-01 05:19:04,2024-01-01,mobile,organic_search,7
4,5,1,7940527645,<NA>,view_item,2024-01-01 05:20:27,2024-01-01,mobile,organic_search,138
5,6,1,7940527645,<NA>,view_item,2024-01-01 05:21:04,2024-01-01,mobile,organic_search,131
6,7,2,2772890305,<NA>,view_item,2024-01-01 05:57:53,2024-01-01,mobile,affiliate,36
7,8,3,7286947083,<NA>,view_item,2024-01-01 06:09:33,2024-01-01,mobile,organic_search,84
8,9,4,7282267126,<NA>,view_item,2024-01-01 07:32:25,2024-01-01,desktop,organic_search,138
9,10,5,2227241223,<NA>,view_item,2024-01-01 07:56:38,2024-01-01,desktop,organic_search,17


In [24]:
profile(user_events, "user_events", ["event_id"])

===== user_events =====

***** Shape *****

Rows: 603272, Columns: 10

***** Dtypes *****

event_id                     int64
session_id                   int64
visitor_id                   int64
customer_id                  Int64
event_type                     str
event_ts            datetime64[us]
event_date_local    datetime64[us]
device                         str
traffic_source                 str
product_id                   Int64
dtype: object

***** Unique Key *****

How many duplicates in event_id? 0
How many missing values in event_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
event_id,0,0.00
session_id,0,0.00
visitor_id,0,0.00
customer_id,509422,84.44
event_type,0,0.00
event_ts,0,0.00
event_date_local,0,0.00
device,0,0.00
traffic_source,11944,1.98
product_id,34616,5.74



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
event_id,603272.0,301636.5,174149.770135,1.0,6033.71,30164.55,150818.75,301636.5,452454.25,573108.45,597239.29,603272.0
session_id,603272.0,104772.981333,60393.950813,1.0,2107.0,10465.0,52485.0,105122.0,157272.25,198909.0,207250.0,209335.0
visitor_id,603272.0,5525060572.708505,2588376993.464117,1000097751.0,1092940140.0,1461158514.0,3308332201.0,5519829106.5,7785308333.5,9550829663.0,9913210932.0,9999973527.0
customer_id,93850.0,2594.888269,1901.668446,1.0,27.0,194.0,969.0,2242.5,3886.0,6247.0,7109.0,7483.0
product_id,568656.0,111.05032,61.835837,1.0,5.0,14.0,59.0,111.0,153.0,222.0,246.0,260.0



***** CATEGORICAL VARIABLES *****


----- event_type: 5 unique -> 5 after strip/lower -----



...

,count
device,
mobile,404744
desktop,168074
tablet,30454



----- traffic_source: 7 unique -> 7 after strip/lower -----



,count
traffic_source,
organic_search,153019
paid_social,117595
direct,102258
paid_search,82785
email,53008
referral,43266
affiliate,39397
NaN,11944



***** DATE RANGE *****



,min,max
event_ts,2024-01-01 05:15:03,2026-09-30 20:56:29
event_date_local,2024-01-01 00:00:00,2026-09-30 00:00:00


## My questions
1. Are all 'event_ts' (UTC) values within the data period stated in the data dictionary?
2. Can one 'session_id' have more than one 'visitor_id', 'device' or 'traffic_source'?
3. For which 'event_type' values is 'customer_id' filled in, and is it always filled in for purchase events?
4. Are there sessions or visitors with unusual activity (very many events, very little time between events) that could be bots?
5. Is 'product_id' filled in only for 'view_item' and 'add_to_cart' events?

### Q1. Are all 'event_ts' (UTC) values within the data period stated in the data dictionary?

**Answer:** Yes (from profile()):

| Column | Min | Max |
|---|---|---|
| event_ts (UTC) | 2024-01-01 05:15 | 2026-09-30 20:56 |
| event_date_local | 2024-01-01 | 2026-09-30 |

All events are inside the data period, both in UTC and in local time.

### Q2. Can one 'session_id' have more than one 'visitor_id', 'device' or 'traffic_source'?

In [25]:
session_check = user_events.groupby("session_id")[["visitor_id", "device", "traffic_source"]].nunique()
display((session_check > 1).sum())

visitor_id        0
device            0
traffic_source    0
dtype: int64

**Answer:** No. Every session has exactly one visitor_id, one device and one traffic_source.

### Q3. For which 'event_type' values is 'customer_id' filled in, and is it always filled in for purchase events?

In [26]:
has_customer = user_events["customer_id"].notna()
customer_share = has_customer.groupby(user_events["event_type"]).mean().round(3)
display(customer_share.to_frame("Share_with_customer_id"))

,Share_with_customer_id
event_type,
add_payment_info,1.000
add_to_cart,0.179
begin_checkout,0.198
purchase,1.000
view_item,0.120


**Answer:**
- customer_id is filled in for all event types, but only partly for browsing: view_item 12%, add_to_cart 18%, begin_checkout 20% (only logged-in visitors).
- It is always filled in (100%) for add_payment_info and purchase, as expected: the customer is identified at payment.

### Q4. Are there sessions or visitors with unusual activity (very many events, very little time between events) that could be bots?

In [27]:
# 1. Events per session: a normal session is short
events_per_session = user_events.groupby("session_id").size().sort_values(ascending=False)
print(f"Median events per session: {events_per_session.median()}")
display(events_per_session.head())

# 2. Look at the longest session
display(user_events[user_events["session_id"] == 161644].head(20))

# 3. Events per session for each visitor
n_sessions = user_events.groupby("visitor_id")["session_id"].nunique()
n_events = user_events.groupby("visitor_id").size()
display((n_events / n_sessions).sort_values(ascending=False).head())

# 4. Median time between events for visitors with many events
ue = user_events.sort_values(["visitor_id", "event_ts"])
ue["gap_sec"] = ue.groupby("visitor_id")["event_ts"].diff().dt.total_seconds()
median_n_events = pd.DataFrame({
    "median_gap_sec": ue.groupby("visitor_id")["gap_sec"].median(),
    "n_events": n_events
})
display(median_n_events[median_n_events["n_events"] > 50].sort_values("median_gap_sec").head())

# 5. Summary of the suspected bot
bot = user_events[user_events["visitor_id"] == 4410023875]
print(f"Sessions: {bot['session_id'].nunique()}, events: {len(bot)}")
display(bot["event_date_local"].agg(["min", "max"]))

Median events per session: 2.0


session_id
161644    152
163612    148
162737    142
159299    141
158531    140
dtype: int64

,event_id,session_id,visitor_id,customer_id,event_type,event_ts,event_date_local,device,traffic_source,product_id
466135,466136,161644,4410023875,<NA>,view_item,2026-04-07 23:30:58,2026-04-08,desktop,direct,39
466136,466137,161644,4410023875,<NA>,view_item,2026-04-07 23:30:59,2026-04-08,desktop,direct,41
466137,466138,161644,4410023875,<NA>,view_item,2026-04-07 23:31:03,2026-04-08,desktop,direct,43
466138,466139,161644,4410023875,<NA>,view_item,2026-04-07 23:31:07,2026-04-08,desktop,direct,44
466139,466140,161644,4410023875,<NA>,view_item,2026-04-07 23:31:08,2026-04-08,desktop,direct,45
466140,466141,161644,4410023875,<NA>,view_item,2026-04-07 23:31:12,2026-04-08,desktop,direct,47
466141,466142,161644,4410023875,<NA>,view_item,2026-04-07 23:31:15,2026-04-08,desktop,direct,48
466142,466143,161644,4410023875,<NA>,view_item,2026-04-07 23:31:19,2026-04-08,desktop,direct,49
466143,466144,161644,4410023875,<NA>,view_item,2026-04-07 23:31:22,2026-04-08,desktop,direct,50
466144,466145,161644,4410023875,<NA>,view_item,2026-04-07 23:31:24,2026-04-08,desktop,direct,51


visitor_id
4410023875    98.0
7141812858    23.0
1291129597    21.0
9654444991    21.0
3801961484    20.0
dtype: float64

,median_gap_sec,n_events
visitor_id,,
4410023875,3.0,2548
6110652584,41.5,51
2201285959,43.0,54
1254507222,43.0,60
2720468496,44.0,69


Sessions: 26, events: 2548


min   2026-03-09
max   2026-04-17
Name: event_date_local, dtype: datetime64[us]

**Answer:** Yes, one visitor (4410023875) looks like a bot → DQ-19.
- Normal session: 2 events. This visitor: 98 events per session.
- Time between events: 3 seconds. Other active visitors: 40+ seconds.
- 2,548 events in 26 sessions, March–April 2026, only product views, at night, no purchases.
- No other visitor looks like this.

### Q5. Is 'product_id' filled in only for 'view_item' and 'add_to_cart' events?

In [28]:
product_filled = user_events.groupby("event_type")["product_id"].count()
print(product_filled)

event_type
add_payment_info         0
add_to_cart          32654
begin_checkout           0
purchase                 0
view_item           536002
Name: product_id, dtype: Int64


**Answer:** Yes. product_id is filled in for all view_item and add_to_cart events, and empty for begin_checkout, add_payment_info and purchase.

## MARKETING_SPEND

In [29]:
marketing_spend.head(10)

,spend_date,channel,campaign_type,impressions,clicks,spend_eur,platform_conversions
0,2024-01-01,affiliate,commission,528,2,0.00,0
1,2024-01-01,email,newsletter_platform,7,6,1.58,0
2,2024-01-01,organic_search,seo_retainer,1076,29,14.52,<NA>
3,2024-01-01,paid_search,brand,7,2,0.27,0
4,2024-01-01,paid_search,non_brand,83,4,1.87,0
5,2024-01-01,paid_social,prospecting,223,2,0.57,0
6,2024-01-01,paid_social,retargeting,68,1,0.38,0
7,2024-01-02,affiliate,commission,606,2,0.00,0
8,2024-01-02,email,newsletter_platform,7,2,1.58,0
9,2024-01-02,organic_search,seo_retainer,911,30,14.52,<NA>


In [30]:
profile(marketing_spend, "marketing_spend", ["spend_date", "channel", "campaign_type"])

===== marketing_spend =====

***** Shape *****

Rows: 7026, Columns: 7

***** Dtypes *****

spend_date              datetime64[us]
channel                            str
campaign_type                      str
impressions                      int64
clicks                           int64
spend_eur                      float64
platform_conversions             Int64
dtype: object

***** Unique Key *****

How many duplicates in spend_date, channel, campaign_type? 4
How many missing values in spend_date, channel, campaign_type? 0
How many fully duplicated rows? 4

***** NULL values in columns *****



,Missing values,% Missing
spend_date,0,0.00
channel,0,0.00
campaign_type,0,0.00
impressions,0,0.00
clicks,0,0.00
spend_eur,0,0.00
platform_conversions,1004,14.29



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
impressions,7026.0,3389.209935,4102.129039,4.0,11.0,19.0,162.25,1952.0,4794.0,11978.5,16854.5,50629.0
clicks,7026.0,48.843723,40.713068,0.0,2.0,7.0,20.0,35.0,66.0,134.0,173.75,454.0
spend_eur,7026.0,13.876937,14.924713,0.0,0.0,0.0,3.19,9.695,20.325,40.4475,65.0075,274.28
platform_conversions,6022.0,1.07456,1.620411,0.0,0.0,0.0,0.0,0.0,2.0,4.0,7.0,17.0



***** CATEGORICAL VARIABLES *****


----- channel: 5 unique -> 5 after strip/lower -----



,count
channel,
paid_social,2012
paid_search,2002
affiliate,1004
email,1004
organic_search,1004



----- campaign_type: 7 unique -> 7 after strip/lower -----



,count
campaign_type,
prospecting,1006
retargeting,1006
commission,1004
newsletter_platform,1004
seo_retainer,1004
brand,1001
non_brand,1001



***** DATE RANGE *****



,min,max
spend_date,2024-01-01,2026-09-30


## My questions
1. Are all 'spend_date' values within the data period stated in the data dictionary?
2. Are 'channel' and 'campaign_type' values written consistently (no different spellings of the same value)?
3. Is 'platform_conversions' empty only for 'organic_search' rows?
4. Does every 'spend_date' have exactly one row for each 'channel' and 'campaign_type' combination (no missing and no duplicate rows)?
5. Are there any rows where 'clicks' are higher than 'impressions'?
6. Does affiliate 'spend_eur' appear only some time after the first affiliate clicks?
7. Is the daily 'spend_eur' for email and organic_search the same on every day within a month?

### Q1. Are all 'spend_date' values within the data period stated in the data dictionary?

**Answer:** Yes (from profile()). spend_date goes from 2024-01-01 to 2026-09-30, same as the data period.

### Q2. Are 'channel' and 'campaign_type' values written consistently (no different spellings of the same value)?

**Answer:** Yes (from profile()). channel has 5 values and campaign_type has 7 values. The number of unique values does not change after strip/lower, so there are no different spellings.

### Q3. Is 'platform_conversions' empty only for 'organic_search' rows?

In [31]:
conv_organic = marketing_spend.groupby("channel")["platform_conversions"].count()
display(conv_organic)

channel
affiliate         1004
email             1004
organic_search       0
paid_search       2002
paid_social       2012
Name: platform_conversions, dtype: Int64

**Answer:** Yes. platform_conversions is empty for all organic_search rows and filled in for all other channels (count equals the number of rows per channel).

### Q4. Does every 'spend_date' have exactly one row for each 'channel' and 'campaign_type' combination (no missing and no duplicate rows)?

In [32]:
#Duplicate rows
date_check = marketing_spend.groupby(["spend_date", "channel", "campaign_type"]).size()
display(date_check[date_check > 1])

#Missing Rows

print(f"spend_date unique: {marketing_spend["spend_date"].nunique()}")
display(marketing_spend.groupby("campaign_type")["spend_date"].nunique().sort_values(ascending = False))

spend_date  channel      campaign_type
2026-02-03  paid_social  prospecting      2
                         retargeting      2
2026-02-04  paid_social  prospecting      2
                         retargeting      2
dtype: int64

spend_date unique: 1004


campaign_type
commission             1004
newsletter_platform    1004
prospecting            1004
seo_retainer           1004
retargeting            1004
brand                  1001
non_brand              1001
Name: spend_date, dtype: int64

**Answer:** No.
- Duplicates: paid_social (prospecting, retargeting) has 2 rows on 2026-02-03 and 2026-02-04 → 4 extra rows → DQ-16.
- Missing: paid_search (brand, non_brand) has 1,001 days instead of 1,004 → 3 days missing, probably a failed export → DQ-17.

### Q5. Are there any rows where 'clicks' are higher than 'impressions'?

In [33]:
click_higher = marketing_spend[marketing_spend["clicks"] > marketing_spend["impressions"]]
print(f"Rows with clicks > impressions: {len(click_higher)}")
print(f"Of them in email channel: {(click_higher['channel'] == 'email').sum()}")

Rows with clicks > impressions: 475
Of them in email channel: 475


**Answer:** Yes, 475 rows, all in the email channel. This is expected: for email, impressions = emails sent that day, while clicks also include clicks on emails sent on earlier days. On days without a newsletter, clicks can be higher than emails sent. Not an error.

### Q6. Does affiliate 'spend_eur' appear only some time after the first affiliate clicks?

In [34]:
aff = marketing_spend[marketing_spend["channel"] == "affiliate"]
print(f"First time clicks > 0:    {aff[aff["clicks"] > 0]["spend_date"].min()}")
print(f"First time spend_eur > 0: {aff[aff["spend_eur"] > 0]["spend_date"].min()}")

First time clicks > 0:    2024-01-01 00:00:00
First time spend_eur > 0: 2024-03-06 00:00:00


**Answer:** Yes. First affiliate clicks: 2024-01-01. First affiliate spend: 2024-03-06 (about 2 months later). This fits the data dictionary: commission is paid only for approved orders, so spend comes later than clicks.

### Q7. Is the daily 'spend_eur' for email and organic_search the same on every day within a month?

In [35]:
y_m_marketing_spend = marketing_spend.copy()
y_m_marketing_spend.insert(1, "year_month", (marketing_spend["spend_date"]).dt.to_period("M"))

daily_spend = y_m_marketing_spend[(y_m_marketing_spend["channel"] == "email") | (y_m_marketing_spend["channel"] == "organic_search")]


daily_spend_unique = daily_spend.groupby(["year_month", "channel"])["spend_eur"].nunique()
print(f"Occurencies: {daily_spend_unique[daily_spend_unique > 1].count()}")



Occurencies: 0


**Answer:** Yes. In every month, email and organic_search have the same spend_eur on every day (1 unique value per channel per month). This fits the data dictionary: fixed monthly fees are spread evenly over the days.

## Profiling summary

Profiling found 19 data issues (see docs/data_issues.md). Below they are grouped by which part of the analysis they would affect if not fixed.

#### Revenue, orders and products
If not fixed, revenue and order counts would be wrong: some orders are counted twice, test orders are included, some prices and quantities are wrong, and profit for 2 products would be negative.
- DQ-01, DQ-07, DQ-08, DQ-09, DQ-10, DQ-11, DQ-12, DQ-13, DQ-18

#### Customers
If not fixed, the number of customers would be too high (duplicate and test accounts), and analysis by city, gender and age would be split or wrong.
- DQ-01, DQ-02, DQ-03, DQ-04, DQ-05, DQ-06

#### Web traffic and conversion
If not fixed, the bot would add fake sessions and lower the conversion rate, the number of purchase events would not match orders (cause still unknown), and some sessions would have no traffic source.
- DQ-14, DQ-15, DQ-19

#### Marketing spend
If not fixed, paid_social spend would be counted twice on 2 days, and paid_search spend would be missing for 3 days.
- DQ-16, DQ-17

DQ-01 is in two groups because it affects both customers and revenue.

#### What looks fine
- **Unique keys:** customers, products, order_items and user_events have unique keys with no missing values. Only orders (DQ-09) and marketing_spend (DQ-16) have duplicate keys.
- **NULLs:** most empty values are expected by design (e.g. session_id, coupon_code, discontinued_date, customer_id and product_id in user_events, platform_conversions for organic_search). Unexpected ones are logged: city (DQ-04), return_reason (DQ-11), traffic_source (DQ-14).
- **Dates:** all dates are inside the data period. promised_date in October 2026 and launch_date before 2024 are normal. Exceptions are logged: test account signup (DQ-01), impossible birth years (DQ-06), delivery before order (DQ-18).

#### Open for cross-table checks (02 notebook)
- Were products sold before launch_date or after discontinued_date?
- Do large quantities (DQ-12) match order_total?
- Why do purchase events not match orders (DQ-15)?
- Does order_total = sum of order lines + shipping_fee?
- Is return_date after delivered_date?